# Train Dr. Ananya (psychologist mode) on Colab
Runtime > Change runtime type > **T4 GPU** (free) or better. Run the cells top to bottom (about 1-3 hours on a T4 with the 3B model).
Nothing is claimed until cell 6 prints the baseline and tuned numbers. Not clinical validation.

In [ ]:
!git clone -q https://github.com/mallikarjunr444-gif/medicus-care.git
%cd medicus-care/ml/ananya_psychologist
!pip -q install datasets peft trl accelerate bitsandbytes transformers requests

## 1. Hugging Face login (the counselling dataset is gated: accept its terms on its Hub page first)

In [ ]:
from huggingface_hub import login
login()  # paste a token with read access (write access too if you want to push the adapter)

## 2. Build the data (add ESConv.json with --esconv if you downloaded it; check licences)

In [ ]:
!python prepare_data.py --out data
!wc -l data/*.jsonl

## 3. Baseline benchmark (before training)

In [ ]:
BASE = "Qwen/Qwen2.5-3B-Instruct"   # use Qwen/Qwen2.5-7B-Instruct on an A100/L4
!python evaluate.py --backend hf --base {BASE} --out baseline_report.json

## 4. Train (LoRA, 4-bit)

In [ ]:
!python train_lora.py --data data --out ananya-lora --base {BASE} --epochs 2

## 5. Benchmark the tuned model (same cases)

In [ ]:
!python evaluate.py --backend hf --base {BASE} --adapter ananya-lora --out tuned_report.json

## 6. Read some transcripts yourself, then save the adapter

In [ ]:
import json
r = json.load(open("tuned_report.json"))
for t in r["transcripts"][:6]:
    print("USER :", t["user"]); print("ANANYA:", t["final"]); print()

In [ ]:
# Optional: push the adapter, then set it as ADAPTER in your Space
# from huggingface_hub import HfApi
# HfApi().create_repo("YOUR_NAME/ananya-lora", exist_ok=True)
# HfApi().upload_folder(folder_path="ananya-lora", repo_id="YOUR_NAME/ananya-lora")